# Project 09: Intelligent PDF Knowledge Assistant (RAG)
### Multi-Document Ingestion, Semantic Retrieval & Exact Citations

Building a verifiable question-answering assistant over proprietary PDF manuals.

In [ ]:
import numpy as np
from dataclasses import dataclass

@dataclass
class PDFChunk:
    text: str
    source_pdf: str
    page: int
    embedding: np.ndarray

# Mock PDF Knowledge Base
np.random.seed(42)
pdf_corpus = [
    PDFChunk("Refund requests must be filed within 30 days of initial invoice date.", "billing_faq.pdf", 3, np.array([0.9, 0.1, 0.2])),
    PDFChunk("Enterprise SLA guarantees 99.99% uptime with 15-minute response times.", "enterprise_sla.pdf", 7, np.array([0.1, 0.85, 0.4])),
    PDFChunk("Database backups execute nightly at 02:00 UTC with 30-day retention.", "security_whitepaper.pdf", 12, np.array([0.3, 0.2, 0.9]))
]

def answer_with_citations(query_vec: np.ndarray, corpus: list):
    scores = []
    for c in corpus:
        sim = float(np.dot(query_vec, c.embedding) / (np.linalg.norm(query_vec) * np.linalg.norm(c.embedding)))
        scores.append((c, sim))
    scores.sort(key=lambda x: x[1], reverse=True)
    top_chunk, best_score = scores[0]
    return {
        "answer": f"According to page {top_chunk.page} of '{top_chunk.source_pdf}': {top_chunk.text}",
        "citation": {"file": top_chunk.source_pdf, "page": top_chunk.page, "similarity": round(best_score, 4)}
    }

query = np.array([0.88, 0.12, 0.25]) # Billing query vector
result = answer_with_citations(query, pdf_corpus)
print("PDF RAG Assistant Response:")
print(f"Answer: {result['answer']}")
print(f"Citation metadata: {result['citation']}")